# Setup

In [57]:
import requests
from bs4 import BeautifulSoup
from typing import List, Optional, TypedDict
from dotenv import load_dotenv
from pydantic import BaseModel, Field, ConfigDict
from langchain.chat_models import init_chat_model


load_dotenv()
llm = init_chat_model('gpt-4.1-mini')

# Schemas

In [58]:
from typing import List, Optional, Union, Literal
from pydantic import BaseModel, Field, ConfigDict

# OpenAI Strict 모드를 위한 공통 설정
STRICT_CONF = ConfigDict(extra='forbid')

# 비판 결과 스키마
class CriticReview(BaseModel):
    model_config = STRICT_CONF
    is_perfect: bool = Field(description="논리적 오류가 전혀 없으면 True")
    critique_points: List[str] = Field(default=[], description="오류 지점 및 수정 요구사항 목록")
    severity: Literal["CRITICAL", "MINOR", "NONE"] = Field(description="오류의 심각도")
    
class OptionSummary(BaseModel):
    model_config = STRICT_CONF
    option_name: str
    brief_description: str # 예: "스타벅스 50% 할인 + 편의점 7% 할인"

class OptionsPreview(BaseModel):
    model_config = STRICT_CONF
    summaries: List[OptionSummary]

# 판단 노드(Decision Node) 전용 스키마
class FlowDecision(BaseModel):
    model_config = STRICT_CONF
    # str 대신 Literal을 사용하여 '객관식'으로 강제합니다.
    decision: Literal["direct", "selection"] = Field(
        description="단일 카드면 'direct', 선택이 필요하면 'selection'"
    )
    options: List[str] = Field(default=[], description="선택형일 경우 후보군 카드 이름 리스트")
    reason: str = Field(description="그렇게 판단한 이유")

# 1. 최하위 계층부터 정의
class PerformanceTier(BaseModel):
    model_config = STRICT_CONF
    tier_min: int
    limit: int

class BenefitLimit(BaseModel):
    model_config = STRICT_CONF
    monthly: Optional[int] = None
    yearly: Optional[int] = None
    monthly_performance_tiers: Optional[List[PerformanceTier]] = None

class BenefitCondition(BaseModel):
    model_config = STRICT_CONF
    min_performance: int = 0
    min_per_transaction: Optional[int] = Field(None, description="혜택을 받기 위한 건당 최소 결제 금액")
    payment_method: str = "ANY"
    is_overseas_only: Optional[bool] = None
    location_exclude: Optional[List[str]] = None
    per_transaction_cap: Optional[int] = None
    platform: Optional[str] = None
    domestic_only: Optional[bool] = None

class PerformanceImpact(BaseModel):
    model_config = STRICT_CONF
    counts_toward_performance: bool
    is_all_or_nothing_exclusion: bool = False
    comment: Optional[str] = None

CardCategory = Literal[
    "FOOD",               # 음식점
    "MEDICAL",            # 병원/약국
    "EDUCATION",          # 학원
    "PARKING_LOT",        # 주차장
    "OIL",                # 주유소
    "TRANSPORTATION",     # 교통 (버스, 지하철, 택시 등)
    "DAILY_LIFE_GROUP",   # 카페, 마트, 편의점, 쇼핑, 문화(영화) 등
    "OVERSEAS",           # 해외 결제
    "TRAVEL_SHOPPING",    # 면세점, 항공 등
    "AIRPORT_LOUNGE",     # 공항 라운지
    "OTHER"               # 기타
]
BenefitType = Literal[
    "PERCENT_DISCOUNT",    # % 할인
    "KRW_DISCOUNT",        # 원 단위 할인
    "CASHBACK",            # 현금 캐시백
    "POINT_ACCUMULATION",  # 포인트 적립
    "FEE_WAIVER",          # 수수료 면제
    "FREE_ACCESS"          # 무료 이용 (라운지 등)
]
BenefitUnit = Literal["PERCENT", "KRW", "COUNT"]

class CardBenefit(BaseModel):
    model_config = ConfigDict(extra='forbid') # 엄격한 모드 유지
    
    benefit_id: str = Field(description="혜택의 고유 ID (예: COFFEE_50)")
    category: CardCategory = Field(description="DB 매핑용 카테고리")
    merchant: Optional[List[str]] = Field(default=None, description="혜택 가맹점 리스트")
    type: BenefitType = Field(description="혜택의 종류 (객관식)")
    value: float = Field(description="혜택의 수치 (10.0, 5000.0 등)")
    unit: BenefitUnit = Field(description="수치의 단위 (PERCENT, KRW, COUNT)")
    
    conditions: 'BenefitCondition'
    limits: 'BenefitLimit'
    performance_impact: 'PerformanceImpact'

# 2. 중간 계층 (Logic 파트)
class PerformanceLogic(BaseModel):
    model_config = STRICT_CONF
    calculation_period: str
    global_performance_exclusion: List[str]
    domestic_only_performance: bool

class GracePeriodLogic(BaseModel):
    model_config = STRICT_CONF
    duration: str
    default_benefit_tier: str
    daily_life_limit: int
    air_dutyfree_limit: int
    transport_limit: int

# 3. 최상위 계층 (Root)
class CardDataSchema(BaseModel):
    model_config = STRICT_CONF
    card_id: str
    card_name: str
    issuer: str
    critical_warning: str
    performance_logic: PerformanceLogic
    benefits: List[CardBenefit]
    grace_period_logic: GracePeriodLogic

In [59]:
from typing import List, Optional, Union, Literal

class State(TypedDict):
    card_id: str
    card_name: Optional[str]
    raw_text: Optional[str]
    decision: Optional[Literal["direct", "selection"]]    
    options_list: List[str]        # AI가 찾아낸 후보군 이름들
    summaries: List[OptionSummary] # 사용자에게 보여줄 패키지별 요약 텍스트
    selected_option: Optional[str] # 사용자가 최종 선택한 패키지명
    critic_feedback: Optional[dict]
    output_json: Optional[dict]
    iteration_count: int = 0
    is_manually_fixed: Optional[bool]

# Nodes

In [60]:
from typing import TypedDict, List, Optional
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate
import json


# 노드 함수 정의

def node_ingest_card_data(state: State):
    card_id = state["card_id"]
    print(f" [Node: Ingest] 카드 ID {card_id} 크롤링 시작...")
    
    url = f"https://api.card-gorilla.com:8080/v1/cards/{card_id}"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36...",
    }
    
    # 실제 요청 수행
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        raise Exception(f"데이터 로드 실패: {response.status_code}")
        
    data = response.json()
    
    # 텍스트 정제 (BeautifulSoup 활용)
    benefit_segments = []
    for benefit in data.get("key_benefit", []):
        soup = BeautifulSoup(benefit.get("info", ""), "html.parser")
        benefit_segments.append(f"[{benefit.get('title')}]\n{soup.get_text(separator=' ').strip()}")
    
    return {
        "card_name": data.get("name"),
        "raw_text": "\n\n".join(benefit_segments)
    }

def node_decide_flow(state: State):
    print("[Node: Decide] 단일 카드인지 선택형인지 판단 중...")
    
    # 파이프(|) 연산자 대신 직접 invoke 호출
    structured_llm = llm.with_structured_output(FlowDecision)
    prompt = ChatPromptTemplate.from_template("""
    이 카드의 상세 텍스트를 보고 다음을 판단해:
    1. '기본 혜택(모든 사용자 공통)'과 '선택 혜택(여러 옵션 중 하나만 택일)'을 구분해.
    2. 만약 특정 이름(예: 라이프스타일 패키지) 아래 1~6번 혹은 A~F형 같은 '패키지'가 존재한다면, 그 패키지들이 진짜 'selection' 후보야.
    3. 단순히 카테고리(대중교통, 통신 등)가 나열된 것은 'selection'이 아니야.

    [텍스트]: {text}
    """)
    
    # 텍스트 채우기 및 호출
    formatted_input = prompt.format(text=state["raw_text"])
    response = structured_llm.invoke(formatted_input)
    
    return {
        "decision": response.decision,
        "options_list": response.options
    }

def node_critic(state: State):
    print("[Node: Critic] 추출된 데이터의 논리 결함을 검토 중...")
    
    critic_llm = llm.with_structured_output(CriticReview)
    
    prompt = ChatPromptTemplate.from_template("""
    너는 카드 약관 분석 전문가이자 깐깐한 감사관이야. 
    [원본 텍스트]와 [추출된 JSON]을 비교해서 논리적 오류를 찾아내.
    특히 '전월 실적 제외 항목'과 '혜택 적용 제외 항목'이 정확히 반영되었는지 독사처럼 감시해.
    performance_impact의 comment 내용과 counts_toward_performance 불리언 값이 서로 모순되지 않는지 반드시 체크해. 
    텍스트로는 포함된다고 하고 값은 제외라고 하면 그건 실패(is_perfect: False)야.
    [원본 텍스트]: {raw_text}
    [추출된 JSON]: {output_json}
    """)
    
    review = critic_llm.invoke(prompt.format(
        raw_text=state["raw_text"],
        output_json=json.dumps(state["output_json"], ensure_ascii=False)
    ))
    
    return {"critic_feedback": review.model_dump()}

def node_extract_data(state: State):
    print(f"[Node: Extract] '{state['card_name']}' 데이터 정밀 분석 및 JSON 생성 중...")
    
    structured_llm = llm.with_structured_output(CardDataSchema)
    
    selected_info = f"사용자가 선택한 옵션: {state.get('selected_option', '기본/단일 패키지')}"
    
    feedback_section = ""
    if state.get("critic_feedback") and not state["critic_feedback"].get("is_perfect"):
        points = "\n- ".join(state["critic_feedback"].get("critique_points", []))
        feedback_section = f"\n\n [비판자 피드백 반영 필요]:\n- {points}"

    # 지시 사항에 Literal 값들을 명시합니다.
    system_msg = f"""너는 신용카드 약관 분석 전문가야. 아래 규칙을 엄격히 준수해.

[데이터 규격 규칙]
- category: FOOD, MEDICAL, EDUCATION, PARKING_LOT, OIL, TRANSPORTATION, DAILY_LIFE_GROUP, OVERSEAS, TRAVEL_SHOPPING, AIRPORT_LOUNGE, OTHER 중 하나만 선택.
- type: PERCENT_DISCOUNT, KRW_DISCOUNT, CASHBACK, POINT_ACCUMULATION, FEE_WAIVER, FREE_ACCESS 중 하나만 선택.
- unit: PERCENT, KRW, COUNT 중 하나만 선택.

[추출 지시]
1. {selected_info}에 해당하는 혜택 정보를 집중적으로 분석해.
2. '전월 실적 제외 항목'을 global_performance_exclusion에 빠짐없이 나열해.
3. 특히 '할인 받은 결제 건이 실적에서 제외되는지' 여부를 performance_impact의 is_all_or_nothing_exclusion에 정확히 기록해.
4. 실적 구간별로 한도가 다르다면 monthly_performance_tiers에 모든 구간을 명시해.{feedback_section}
5. '건당 최소 결제 금액' 조건이 있는지 텍스트에서 눈을 부릅뜨고 찾아내어 min_per_transaction에 기록해.
6. '혜택 제외 가맹점'(백화점, 대형마트, 역사 내 매장 등)은 반드시 conditions.location_exclude에 리스트로 넣어.
7. '무이자 할부 시 혜택 제외' 여부를 확인하여 critical_warning에 반드시 포함시켜.
"""
    
    result = structured_llm.invoke([
        {"role": "system", "content": system_msg},
        {"role": "user", "content": f"카드 원문 텍스트:\n{state['raw_text']}"}
    ])
    
    result_dict = result.model_dump()
    result_dict['card_id'] = state['card_id']
    
    # iteration_count를 1 증가시켜 반환합니다.
    return {"output_json": result_dict, "iteration_count": state.get("iteration_count", 0) + 1}


def node_summarize_options(state: State):
    print("[Node: Summarize] 선택지별 맛보기 정보 생성 중...")
    
    prompt = ChatPromptTemplate.from_template("""
    사용자에게 보여줄 선택지 요약을 만들어줘. 
    특히 '라이프스타일 패키지'처럼 번호가 매겨진 옵션이 있다면, 각 번호별로 혜택이 어떻게 다른지 대조해서 보여줘야 해.

    예: 패키지 1(스타벅스 50%, 오픈마켓 7%), 패키지 2(스타벅스 30%, 오픈마켓 7%)...

    [후보군]: {options_list}
    [텍스트]: {raw_text}
    """)
    
    summarizer = llm.with_structured_output(OptionsPreview)
    res = summarizer.invoke(prompt.format(
        options_list=state["options_list"],
        raw_text=state["raw_text"]
    ))
    
    # 요약 정보를 State에 저장 (State에 summaries 필드 추가 필요)
    return {"summaries": res.summaries}

from langgraph.types import interrupt

def node_wait_user_choice(state: State):
    # 1. 요약된 내용 출력 (summaries가 문자열이라면 그대로 출력)
    print("\n[AI 요약 리포트]")
    for i, s in enumerate(state["summaries"], 1):
            print(f"{i}. {s.option_name}: {s.brief_description}")    
    # 2. 인터럽트 발생 (사용자 입력 대기)
    # CLI 환경이라면 여기서 멈추고 입력을 기다립니다.
    user_input = interrupt({
        "question": "분석할 패키지 번호(1~6)나 이름을 입력해주세요.",
        "options": state["options_list"]
    })
    
    # 3. 입력값 처리 (번호인지 이름인지 판단)
    selected = None
    
    # 입력이 숫자(1, 2, 3...)인 경우 처리
    if str(user_input).isdigit():
        idx = int(user_input) - 1
        if 0 <= idx < len(state["options_list"]):
            selected = state["options_list"][idx]
            print(f"{idx+1}번({selected})이 선택되었습니다.")
    
    # 입력이 이름인 경우 처리
    if not selected:
        selected = user_input
        print(f"'{selected}' 패키지가 선택되었습니다.")
    
    # 4. 다음 단계를 위해 State 업데이트
    # decision을 'direct'로 바꿔야 추출 노드로 넘어갑니다.
    return {
        "decision": "direct", 
        "selected_option": selected,
        "card_name": f"{state['card_name']} ({selected})"
    }

def node_admin_review(state: State):
    print("\n[Node: Admin Review] AI가 데이터 추출에 실패했습니다. 관리자 개입이 필요합니다.")
    print(f"비판자 지적 사항: {state['critic_feedback'].get('critique_points')}")
    
    # 관리자에게 수정된 JSON 입력을 요청 (interrupt 사용)
    # 실제 환경에서는 UI를 통해 JSON 편집기를 띄워주는 역할을 합니다.
    fixed_json = interrupt({
        "message": "AI가 추출에 실패한 항목을 직접 수정하여 입력해주세요.",
        "current_json": state["output_json"],
        "error_reasons": state["critic_feedback"].get("critique_points")
    })
    
    # 관리자가 수정한 데이터로 업데이트하고 종료 단계로 보냅니다.
    return {
        "output_json": fixed_json, 
        "is_manually_fixed": True # 사람이 직접 수정했다는 플래그 (State에 추가 권장)
    }

import os
import psycopg2
from psycopg2.extras import Json
from typing import Any, Dict

def node_db_commit(state: Dict[str, Any]):
    print(f"[Node: DB Commit] 카드 ID '{state['card_id']}' 저장 시작...")

    # 1. 환경 변수에서 AWS DB 접속 정보 로드 (보안 권장)
    # 직접 입력 시: host="your-rds-endpoint.aws.com", user="admin" 등
    db_config = {
            "host": os.getenv("DB_HOST"),
            "dbname": os.getenv("DB_NAME"),
            "user": os.getenv("DB_USER"),
            "password": os.getenv("DB_PASSWORD"),
            "port": os.getenv("DB_PORT", "5432") # 기본값 설정 가능
        }

    # 2. 저장할 데이터 추출
    out = state["output_json"]
    card_id = state["card_id"]
    card_company = out.get("issuer")
    card_name = out.get("card_name")
    card_type = state.get("card_type", "CHECK") # 기본값 CHECK

    try:
        with psycopg2.connect(**db_config) as conn:
            with conn.cursor() as cur:
                # 3. UPSERT 쿼리 실행 (따옴표 제거 및 소문자 스네이크 케이스)
                # card_id 컬럼에 UNIQUE 제약 조건이 있어야 ON CONFLICT가 작동합니다.
                query = """
                INSERT INTO card_master (
                    card_id, card_company, card_name, card_type, benefits_json
                ) VALUES (%s, %s, %s, %s, %s)
                ON CONFLICT (card_id) 
                DO UPDATE SET 
                    card_company = EXCLUDED.card_company,
                    card_name = EXCLUDED.card_name,
                    card_type = EXCLUDED.card_type,
                    benefits_json = EXCLUDED.benefits_json,
                    created_at = CURRENT_TIMESTAMP;
                """
                
                cur.execute(query, (
                    card_id, 
                    card_company, 
                    card_name, 
                    card_type, 
                    Json(out) # 딕셔너리를 PostgreSQL jsonb 포맷으로 변환
                ))
                # with문 종료 시 자동 commit
                print(f"DB 저장 완료: {card_company} {card_name} ({card_id})")

    except Exception as e:
        print(f"DB 저장 중 오류 발생: {e}")
        # 필요 시 에러를 다시 raise하여 그래프 흐름 제어
        raise e

    return {"is_committed": True}



# 3. 라우팅 함수 (Conditional Edge용)
def router(state: State):
    if state["decision"] == "selection":
        return "선택필요"
    return "즉시추출"

def critic_router(state: State):
    feedback = state.get("critic_feedback")
    iteration = state.get("iteration_count", 0)

    # 1. 완벽하면 통과
    if feedback and feedback.get("is_perfect"):
        return "통과"

    # 2. 완벽하지 않아도 심각도가 MINOR이고 2번 이상 시도했다면 일단 저장 (관용)
    if feedback and feedback.get("severity") == "MINOR" and iteration >= 2:
        print("[Router] ⚠️ 사소한 오류가 있으나 2회 이상 시도하여 통과시킵니다.")
        return "통과"

    # 3. 3회 이상 실패하면 관리자에게
    if iteration >= 3:
        return "관리자개입"

    return "재수정"


import os
import psycopg2
from psycopg2.extras import Json
from sshtunnel import SSHTunnelForwarder
from dotenv import load_dotenv

# .env 로드
load_dotenv()

def node_db_commit(state):
    print(f"🔒 [SSH Tunneling] RDS 접속을 위해 터널을 개방합니다...")

    # 1. SSH 및 RDS 설정값 불러오기
    ssh_host = os.getenv("SSH_HOST")
    ssh_port = int(os.getenv("SSH_PORT", 22))
    ssh_user = os.getenv("SSH_USER")
    ssh_key = os.getenv("SSH_KEY_PATH") # .pem 파일 경로

    rds_host = os.getenv("RDS_HOST")
    rds_port = int(os.getenv("RDS_PORT", 5432))
    rds_user = os.getenv("RDS_USER")
    rds_password = os.getenv("RDS_PASSWORD")
    rds_db = os.getenv("RDS_DB_NAME")

    # 2. 데이터 준비
    out = state["output_json"]
    card_id = state["card_id"]
    




    # SSH 터널 생성 및 연결
    try:
        
        with SSHTunnelForwarder(
            (ssh_host, ssh_port),
            ssh_username=ssh_user,
            ssh_pkey=ssh_key,
            remote_bind_address=(rds_host, rds_port)
        ) as tunnel:
            print(f"터널 개방 성공! (Local Port: {tunnel.local_bind_port})")

            # 3. 터널을 통한 DB 연결 (host는 localhost, port는 tunnel의 로컬 포트 사용)
            with psycopg2.connect(
                host='127.0.0.1',
                port=tunnel.local_bind_port,
                user=rds_user,
                password=rds_password,
                dbname=rds_db
            ) as conn:
                with conn.cursor() as cur:
                    query = """
                    INSERT INTO card_master (card_id, card_company, card_name, card_type, benefits_json)
                    VALUES (%s, %s, %s, %s, %s)
                    ON CONFLICT (card_id) 
                    DO UPDATE SET 
                        card_company = EXCLUDED.card_company,
                        card_name = EXCLUDED.card_name,
                        card_type = EXCLUDED.card_type,
                        benefits_json = EXCLUDED.benefits_json,
                        created_at = CURRENT_TIMESTAMP;
                    """
                    cur.execute(query, (
                        card_id, 
                        out.get("issuer"), 
                        out.get("card_name"), 
                        state.get("card_type", "CHECK"), 
                        Json(out)
                    ))
                    conn.commit()
                    print(f"✅ DB 저장 성공 (ID: {card_id})")




    except Exception as e:
        print(f"❌ 접속 실패: {e}")
        raise e

    return {"is_committed": True}


# Graph

In [61]:
from langgraph.checkpoint.memory import MemorySaver

# 그래프 조립 
graph = StateGraph(State)

# 1. 노드 등록 (기존 동일)
graph.add_node("수집", node_ingest_card_data)
graph.add_node("판단", node_decide_flow)
graph.add_node("요약", node_summarize_options)
graph.add_node("선택", node_wait_user_choice)
graph.add_node("추출", node_extract_data)
graph.add_node("DB저장", node_db_commit) 

# 2. 엣지 연결 (비판 노드 무시 흐름)
graph.add_edge(START, "수집")
graph.add_edge("수집", "판단")

# 3. 조건부 분기 (단일형 vs 선택형)
graph.add_conditional_edges(
    "판단",
    router,
    {
        "선택필요": "요약",
        "즉시추출": "추출"
    }
)

# 선택형 흐름: 요약 -> 사용자 선택 -> 추출 -> DB저장
graph.add_edge("요약", "선택")
graph.add_edge("선택", "추출") 

# 🔥 핵심 수정: 비판(Critic)을 건너뛰고 바로 DB저장으로 연결
graph.add_edge("추출", "DB저장")

# 4. 종료
graph.add_edge("DB저장", END)

# 6. 컴파일
memory = MemorySaver()
app = graph.compile(checkpointer=memory)

# Run

In [62]:
# 1. 실행 설정 정의
config = {"configurable": {"thread_id": "card_analysis_session_03"}}

if __name__ == "__main__":
    initial_input = {"card_id": "2929", "options_list": []}
    
    # 2. invoke 호출 시 config를 함께 넘김
    # '선택' 노드에서 멈춤
    final_state = app.invoke(initial_input, config=config)
    
    # interrupt가 발생하면 여기서 실행이 멈춤.
    if final_state.get("output_json"):
        print("✅ 분석 완료!")

 [Node: Ingest] 카드 ID 2929 크롤링 시작...
[Node: Decide] 단일 카드인지 선택형인지 판단 중...
[Node: Summarize] 선택지별 맛보기 정보 생성 중...

[AI 요약 리포트]
1. A팩: OTT 50% 할인(월 최대 5,000원), APP 스토어 30% 할인(월 최대 5,000원), 노래방·PC방 20% 할인(월 최대 2,000원), 택시·철도 20% 할인(월 최대 2,000원), 편의점 20% 할인(월 최대 2,000원, KB Pay 이용 시), 영화관 4,000원 할인 (1회, 조건 충족 시)
2. B팩: 쇼핑 멤버십 50% 할인(월 최대 5,000원), 통신요금 30% 할인(월 최대 5,000원), 온라인 패션/라이프 20% 할인(월 최대 2,000원), 배달앱 20% 할인(월 최대 2,000원), 편의점 20% 할인(월 최대 2,000원, KB Pay 이용 시), 레스토랑 및 놀이공원 4,000원 할인(1회, 조건 충족 시)


In [ ]:
import json

# 분석이 끝났는지 확인하고 출력
if "output_json" in final_state and final_state["output_json"]:
    print(f"✅ [{final_state['card_name']}] 분석 완료!")
    print(json.dumps(final_state["output_json"], indent=2, ensure_ascii=False))
else:
    print("❌ 아직 결과가 생성되지 않았습니다. 로그를 확인해 주세요.")

In [63]:
from langgraph.types import Command

# 사용자가 입력하고 싶은 번호 (예: "1")
user_input = "2" 

# 멈춘 지점부터 다시 시작 (동일한 config 사용)
final_result = app.invoke(Command(resume=user_input), config=config)

# 최종 결과물 출력
if final_result.get("output_json"):
    print("\n✅ 최종 분석이 완료되었습니다!")
    import json
    print(json.dumps(final_result["output_json"], indent=2, ensure_ascii=False))

Deserializing unregistered type __main__.OptionSummary from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'OptionSummary')]
Deserializing unregistered type __main__.OptionSummary from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'OptionSummary')]



[AI 요약 리포트]
1. A팩: OTT 50% 할인(월 최대 5,000원), APP 스토어 30% 할인(월 최대 5,000원), 노래방·PC방 20% 할인(월 최대 2,000원), 택시·철도 20% 할인(월 최대 2,000원), 편의점 20% 할인(월 최대 2,000원, KB Pay 이용 시), 영화관 4,000원 할인 (1회, 조건 충족 시)
2. B팩: 쇼핑 멤버십 50% 할인(월 최대 5,000원), 통신요금 30% 할인(월 최대 5,000원), 온라인 패션/라이프 20% 할인(월 최대 2,000원), 배달앱 20% 할인(월 최대 2,000원), 편의점 20% 할인(월 최대 2,000원, KB Pay 이용 시), 레스토랑 및 놀이공원 4,000원 할인(1회, 조건 충족 시)
2번(B팩)이 선택되었습니다.
[Node: Extract] 'KB Youth Club 체크카드 (B팩)' 데이터 정밀 분석 및 JSON 생성 중...
🔒 [SSH Tunneling] RDS 접속을 위해 터널을 개방합니다...
터널 개방 성공! (Local Port: 56796)
✅ DB 저장 성공 (ID: 2929)

✅ 최종 분석이 완료되었습니다!
{
  "card_id": "2929",
  "card_name": "KB Youth Club 체크카드 B팩",
  "issuer": "KB국민카드",
  "critical_warning": "무이자 할부 시 혜택 제외에 대한 내용이 별도로 명시되어 있지 않음.",
  "performance_logic": {
    "calculation_period": "전월 1일 ~ 말일(승인시점 기준)",
    "global_performance_exclusion": [
      "포인트리 충전금액",
      "상품권 및 선불카드 구입·충전 금액",
      "아파트관리비",
      "정부지원금(보육료, 유치원보조비, 바우처 이용금액 등)",
      "대학(원)등록금",
      "국세",
      "지방세",
      "4

# Vector DB 저장용

In [ ]:
import os
import requests
import psycopg2
from bs4 import BeautifulSoup
from openai import OpenAI
from sshtunnel import SSHTunnelForwarder
from dotenv import load_dotenv

load_dotenv()

# 1. OpenAI 클라이언트 설정
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def get_embedding(text):
    # 최신 가성비 모델인 text-embedding-3-small 사용
    response = client.embeddings.create(
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

def store_card_to_vector_db(card_id):
    # --- API 요청 파트 ---
    url = f"https://api.card-gorilla.com:8080/v1/cards/{card_id}"
    headers = {"User-Agent": "Mozilla/5.0", "Accept": "application/json"}
    
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print("API 호출 실패")
        return

    data = response.json()
    card_name = data.get("card_name") or data.get("name") or "알 수 없는 카드"

    # 1. 원문 텍스트를 모두 하나로 병합 (카드사 단서 확보용)
    all_text_for_search = ""
    for benefit in data.get("key_benefit", []):
        raw_info = benefit.get("info", "")
        soup = BeautifulSoup(raw_info, "html.parser")
        all_text_for_search += soup.get_text(separator=" ").strip() + " "

    # 2. LLM에게 추측이 아닌 '텍스트 내 팩트 추출'을 강제 지시
    print(f"원문 텍스트에서 [{card_name}]의 정확한 카드사 정보 추출 중...")
    completion = client.chat.completions.create(
        model="gpt-4.1-mini", 
        messages=[
            {
                "role": "system", 
                "content": "당신은 팩트 기반 정보 추출 전문가입니다. 제공된 신용카드 약관 텍스트를 꼼꼼히 읽고, 해당 카드를 발급한 '카드회사 이름(예: KB국민카드, 신한카드, 현대카드 등)'만 찾아내어 단답형으로 출력하세요. 이름만 보고 임의로 유추하는 것을 엄격히 금지합니다. 텍스트 내에서 증거를 찾을 수 없다면 반드시 '알 수 없는 회사'라고 출력하세요."
            },
            {
                "role": "user", 
                "content": f"카드 원문 텍스트: {all_text_for_search[:3000]}" # 핵심 약관은 대부분 앞부분에 있으므로 3000자 제한으로 비용 절감
            }
        ],
        temperature=0 # 창의성을 완전히 끄고 정확성만 100%로 설정
    )
    card_company = completion.choices[0].message.content.strip()
    
    # 환경 변수 로드
    ssh_host = os.getenv("SSH_HOST")
    rds_host = os.getenv("RDS_HOST")

    print(f"[{card_name}] 벡터화 및 저장 시작...")

# 16개 카테고리 키워드 정의
    keywords_map = {
        "FOOD": ["음식점", "식당", "외식", "패밀리레스토랑"],
        "DELIVERY": ["배달", "배달의민족", "요기요", "쿠팡이츠"],
        "CAFE_BAKERY": ["카페", "스타벅스", "베이커리", "커피", "디저트", "투썸", "이디야", "파리바게뜨", "뚜레쥬르"],
        "MEDICAL": ["병원", "약국", "치과", "한의원", "의료", "건강검진"],
        "EDUCATION": ["학원", "교육", "학습지", "서점", "도서", "강의", "유치원"],
        "PARKING_LOT": ["주차장", "주차", "발레파킹"],
        "OIL": ["주유", "GSCALTEX", "S-OIL", "현대오일뱅크", "SK에너지", "충전소"],
        "TRANSPORTATION": ["대중교통", "버스", "지하철", "택시", "철도", "KTX", "SRT", "K-패스"],
        "TELECOM_UTILITY": ["통신", "SKT", "KT", "LGU+", "이동통신", "공과금", "관리비", "가스비", "전기요금"],
        "CONVENIENCE": ["편의점", "CU", "GS25", "세븐일레븐", "이마트24", "다이소", "올리브영"],
        "SHOPPING": ["마트", "이마트", "홈플러스", "롯데마트", "백화점", "아울렛", "쇼핑", "쿠팡", "온라인쇼핑", "오픈마켓", "11번가"],
        "CULTURE_ENTERTAINMENT": ["영화", "CGV", "메가박스", "롯데시네마", "문화", "공연", "전시", "테마파크", "놀이공원"],
        "SUBSCRIPTION": ["구독", "넷플릭스", "유튜브", "프리미엄", "디즈니플러스", "멜론", "스트리밍"],
        "OVERSEAS": ["해외", "직구", "가맹점(해외)", "아마존", "알리", "테무"],
        "TRAVEL": ["항공", "항공권", "면세점", "호텔", "숙박", "여행사", "렌터카", "야놀자", "여기어때"],
        "AIRPORT_LOUNGE": ["라운지", "공항서비스", "마티나", "스카이허브", "인천공항", "공항발레파킹", "더라운지", "라운지키"]
    }

    try:
        # SSH 터널은 한 번
        with SSHTunnelForwarder(
            (ssh_host, 22),
            ssh_username=os.getenv("SSH_USER"),
            ssh_pkey=os.getenv("SSH_KEY_PATH"),
            remote_bind_address=(rds_host, 5432)
        ) as tunnel:
            
            # DB 연결도 한 번만
            with psycopg2.connect(
                host='127.0.0.1',
                port=tunnel.local_bind_port,
                user=os.getenv("RDS_USER"),
                password=os.getenv("RDS_PASSWORD"),
                dbname=os.getenv("RDS_DB_NAME")
            ) as conn:
                
                with conn.cursor() as cur:
                    for benefit in data.get("key_benefit", []):
                        title = benefit.get("title", "")
                        raw_info = benefit.get("info", "")
                        
                        # HTML 정제
                        soup = BeautifulSoup(raw_info, "html.parser")
                        clean_info = soup.get_text(separator=" ").strip()
                        
                        # 카테고리 분류 로직
                        # 1. 1차 필터링: '제외' 단어에 낚이지 않도록 혜택의 '제목(title)'에서 먼저 키워드를 찾음
                        found_categories = [
                            cat for cat, kws in keywords_map.items() 
                            if any(kw in title for kw in kws)
                        ]
                        
                        # 2. 2차 필터링: 제목에 키워드가 정 없다면, 본문(clean_info)을 검색
                        if not found_categories:
                            found_categories = [
                                cat for cat, kws in keywords_map.items() 
                                if any(kw in clean_info for kw in kws)
                            ]

                        # 3. 단일 맵핑: 파이프(|)로 묶지 않고, 발견된 것 중 1순위 대표 카테고리 딱 1개만 지정
                        final_category = found_categories[0] if found_categories else "OTHER"

                        # 벡터화 및 저장
                        full_text = f"카드명: {card_name} | 상세: {clean_info}"
                        embedding = get_embedding(full_text)

                        cur.execute("""
                            INSERT INTO card_benefit_vectors 
                            (card_id, card_company, card_name, benefit_category, content, embedding)
                            VALUES (%s, %s, %s, %s, %s, %s)
                        """, (
                            card_id, card_company, card_name, 
                            final_category, full_text, embedding
                        ))
                    
                    conn.commit()
                    print(f"[{card_name}] 저장 완료!")

    except Exception as e:
        print(f"오류 발생: {e}")

In [67]:
store_card_to_vector_db("2929")




🤖 원문 텍스트에서 [KB Youth Club 체크카드]의 정확한 카드사 정보 추출 중...
🧬 [KB Youth Club 체크카드] 벡터화 및 저장 시작...
[KB Youth Club 체크카드] 저장 완료!
